In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from sklearn.model_selection import train_test_split
%matplotlib inline

np.random.seed(1)

# Little boilerplate code to find out if we have a gpu
device = 'cpu'
if torch.cuda.device_count() > 0 and torch.cuda.is_available():
    print("Cuda installed! Running on GPU!")
    device = 'cuda'
else:
    print("No GPU available!")
print(f'Device: {device}')

Cuda installed! Running on GPU!
Device: cuda


P-wave picking

In [ ]:
# Loading data
Seismic_Data = np.load(r'D:\KAUST\semster2\ML_IN_GEO\SiesmicEventsDetectionPicking_Normalized.npz',allow_pickle=True) # Change the path to your local path
data = Seismic_Data['data']
label = Seismic_Data['label']
Pwave = Seismic_Data['P']
Swave = Seismic_Data['S']
Endcoda = Seismic_Data['E']

k=67
# Checking the label
if label[k]==0:
    print('It is a noise waveform')
else:
    print('It is a seismic event waveform')
    
print('The data and label shapes are:',data.shape,label.shape)

It is a seismic event waveform
The data and label shapes are: (1000, 6000) (1000,)


Preparing Label Data for Picking

In [3]:
Lab_Detection = np.zeros_like(data)
halfwidth = 40
dim = Lab_Detection.shape[1]

for i,j in enumerate(label):
    #Label for seismic event waveform
    if j==1:
        spt = Pwave[i]
        if spt and (spt-halfwidth >= 0) and (spt+halfwidth < dim):
            Lab_Detection[i, spt-halfwidth:spt+halfwidth] = np.exp(-(np.arange(spt-halfwidth,spt+halfwidth)-spt)**2/(2*(10)**2))[:dim-(spt-halfwidth)]                
        elif spt and (spt-halfwidth < dim):
            Lab_Detection[i, 0:spt+halfwidth] = np.exp(-(np.arange(0,spt+halfwidth)-spt)**2/(2*(10)**2))[:dim-(spt-halfwidth)]

    #Label for noise waveform
    # Nothingto do since the label should be zero

Data preparation

In [4]:
import importlib
import data_utils  
importlib.reload(data_utils)

from data_utils import create_dataloaders
train_loader, test_loader = create_dataloaders(
        data,
        Lab_Detection,
        batch_size=8,
        test_size=0.1,
        random_state=42,
        shuffle_train=True,
        transform=None)

Create Model Class

In [6]:
from model import MLP

input_dim = data.shape[1]
output_dim = data.shape[1]

models = {
    "Model1_ReLU": MLP(
        input_dim=input_dim,
        hidden_layers=[2000],
        activation="relu",
        output_dim=output_dim
    ),

    "Model2_ReLU": MLP(
        input_dim=input_dim,
        hidden_layers=[2000, 3000],
        activation="relu",
        output_dim=output_dim
    ),

    "Model3_ReLU": MLP(
        input_dim=input_dim,
        hidden_layers=[2000, 3000, 4000],
        activation="relu",
        output_dim=output_dim
    ),

    "Model4_LeakyReLU": MLP(
        input_dim=input_dim,
        hidden_layers=[2000, 3000, 4000],
        activation="leakyrelu",
        output_dim=output_dim
    ),

    "Model5_GELU": MLP(
        input_dim=input_dim,
        hidden_layers=[2000, 3000, 4000],
        activation="gelu",
        output_dim=output_dim
    ),
}

Training and evaluating

In [7]:
from data_utils import train_model_picking
import pandas as pd

results = []

for name, model in models.items():
    print(f"Training {name}")
    metrics = train_model_picking(model, train_loader, test_loader, epochs=100)
    metrics["Model"] = name
    results.append(metrics)

results_df = pd.DataFrame(results)
print(results_df)

Training Model1_ReLU
epoch: 0. Loss: 0.01105117704719305
epoch: 1. Loss: 0.010435603559017181
epoch: 2. Loss: 0.005508821923285723
epoch: 3. Loss: 0.002922253217548132
epoch: 4. Loss: 0.010231158696115017
epoch: 5. Loss: 0.0012401052517816424
epoch: 6. Loss: 0.002356802811846137
epoch: 7. Loss: 0.0036083555314689875
epoch: 8. Loss: 0.0038465021643787622
epoch: 9. Loss: 0.0024780863896012306
epoch: 10. Loss: 0.0023256961721926928
epoch: 11. Loss: 0.0011053875787183642
epoch: 12. Loss: 0.0011259660823270679
epoch: 13. Loss: 0.004397101700305939
epoch: 14. Loss: 8.530762897862587e-06
epoch: 15. Loss: 0.0025299456901848316
epoch: 16. Loss: 0.003322835313156247
epoch: 17. Loss: 0.0031037647277116776
epoch: 18. Loss: 0.002940686419606209
epoch: 19. Loss: 0.0025588320568203926
epoch: 20. Loss: 0.0027530821971595287
epoch: 21. Loss: 0.0038185662124305964
epoch: 22. Loss: 0.005581890698522329
epoch: 23. Loss: 0.014427962712943554
epoch: 24. Loss: 0.013338879682123661
epoch: 25. Loss: 0.01796221

In [8]:
results_df

,TP,FP,TN,FN,Precision,Recall,F1-score,Model
0,32,32,18,44,0.500000,0.421053,0.457143,Model1_ReLU
1,37,31,19,42,0.544118,0.468354,0.503401,Model2_ReLU
2,23,31,19,47,0.425926,0.328571,0.370968,Model3_ReLU
3,20,26,24,47,0.434783,0.298507,0.353982,Model4_LeakyReLU
4,29,23,27,45,0.557692,0.391892,0.460317,Model5_GELU


Save models

In [9]:
import os

save_dir = "saved_models_picking"
os.makedirs(save_dir, exist_ok=True)

torch.save(models["Model1_ReLU"].state_dict(), f"{save_dir}/model1.pth")
torch.save(models["Model2_ReLU"].state_dict(), f"{save_dir}/model2.pth")
torch.save(models["Model3_ReLU"].state_dict(), f"{save_dir}/model3.pth")
torch.save(models["Model4_LeakyReLU"].state_dict(), f"{save_dir}/model4.pth")
torch.save(models["Model5_GELU"].state_dict(), f"{save_dir}/model5.pth")

Q2 Transform data

In [19]:
from data_utils import train_model, wavelet_transform, fft_transform, stft_transform, spectrogram_transform
import importlib
import data_utils  
importlib.reload(data_utils)
from data_utils import train_model_picking

results2 = {}

###################################### Wavelet  ################################
train_loader_wav, test_loader_wav = create_dataloaders(
        data,
        Lab_Detection,
        batch_size=8,
        test_size=0.1,
        random_state=42,
        shuffle_train=True,
        transform=wavelet_transform)

model_wav = MLP(
    input_dim=train_loader_wav.dataset.data.shape[1],
    hidden_layers=[2000, 3000],
    activation="relu",
    output_dim=output_dim
)

results2["Wavelet"] = train_model_picking(model_wav, train_loader_wav, test_loader_wav, epochs=100)


####################################### FFT  ################################
train_loader_fft, test_loader_fft = create_dataloaders(
        data,
        Lab_Detection,
        batch_size=8,
        test_size=0.1,
        random_state=42,
        shuffle_train=True,
        transform=fft_transform)

model_fft = MLP(
    input_dim=train_loader_fft.dataset.data.shape[1],
    hidden_layers=[2000, 3000],
    activation="relu",
    output_dim=output_dim
)

results2["FFT"] = train_model_picking(
    model_fft,
    train_loader_fft,
    test_loader_fft,
    epochs=100,
)



###################################### STFT  ################################
train_loader_stft, test_loader_stft = create_dataloaders(
        data,
        Lab_Detection,
        batch_size=8,
        test_size=0.1,
        random_state=42,
        shuffle_train=True,
        transform=stft_transform)

model_stft = MLP(
    input_dim=train_loader_stft.dataset.data.shape[1],
    hidden_layers=[2000, 3000],
    activation="relu",
    output_dim=output_dim
)

results2["STFT"] = train_model_picking(
    model_stft,
    train_loader_stft,
    test_loader_stft,
    epochs=100
)

###################################### ADD Spectrogram (optional)  ################################


train_loader_spec, test_loader_spec = create_dataloaders(
        data,
        Lab_Detection,
        batch_size=8,
        test_size=0.1,
        random_state=42,
        shuffle_train=True,
        transform=spectrogram_transform)

model_spec = MLP(
    input_dim=train_loader_spec.dataset.data.shape[1],
    hidden_layers=[2000, 3000],
    activation="relu",
    output_dim=output_dim
)

results2["spectrogram"] = train_model_picking(
    model_spec,
    train_loader_spec,
    test_loader_spec,
    epochs=100
)

epoch: 0. Loss: 0.027567215263843536
epoch: 1. Loss: 0.01338784396648407
epoch: 2. Loss: 0.01282033696770668
epoch: 3. Loss: 0.004769194405525923
epoch: 4. Loss: 0.0016056846361607313
epoch: 5. Loss: 0.001207580673508346
epoch: 6. Loss: 0.0031494502909481525
epoch: 7. Loss: 0.0010525407269597054
epoch: 8. Loss: 0.0009990133112296462
epoch: 9. Loss: 0.00100063590798527
epoch: 10. Loss: 0.0010185653809458017
epoch: 11. Loss: 0.0039098383858799934
epoch: 12. Loss: 0.003910652361810207
epoch: 13. Loss: 0.0019923720974475145
epoch: 14. Loss: 0.0029401727952063084
epoch: 15. Loss: 0.003016440896317363
epoch: 16. Loss: 0.0009757420048117638
epoch: 17. Loss: 0.0019512630533427
epoch: 18. Loss: 0.0009784954600036144
epoch: 19. Loss: 0.0029211940709501505
epoch: 20. Loss: 0.0030470537021756172
epoch: 21. Loss: 0.0019473271677270532
epoch: 22. Loss: 0.00196734513156116
epoch: 23. Loss: 0.0029195912647992373
epoch: 24. Loss: 2.6120392249140423e-07
epoch: 25. Loss: 0.0009949877858161926
epoch: 26. 

d:\anaconda3\envs\torch_env\Lib\site-packages\torch\functional.py:704: UserWarning: A window was not provided. A rectangular window will be applied,which is known to cause spectral leakage. Other windows such as torch.hann_window or torch.hamming_window can are recommended to reduce spectral leakage.To suppress this warning and use a rectangular window, explicitly set `window=torch.ones(n_fft, device=<device>)`. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\SpectralOps.cpp:842.)
  return _VF.stft(  # type: ignore[attr-defined]


epoch: 0. Loss: 0.5454065799713135
epoch: 1. Loss: 0.7466417551040649
epoch: 2. Loss: 0.7055370211601257
epoch: 3. Loss: 0.5715959668159485
epoch: 4. Loss: 0.5660483241081238
epoch: 5. Loss: 0.5377694368362427
epoch: 6. Loss: 0.5240678787231445
epoch: 7. Loss: 0.4333333373069763
epoch: 8. Loss: 0.7978305816650391
epoch: 9. Loss: 0.4893699586391449
epoch: 10. Loss: 0.6148024201393127
epoch: 11. Loss: 0.6422055959701538
epoch: 12. Loss: 0.5240678787231445
epoch: 13. Loss: 0.466562420129776
epoch: 14. Loss: 0.5715959668159485
epoch: 15. Loss: 0.6275976300239563
epoch: 16. Loss: 0.5368630886077881
epoch: 17. Loss: 0.5938060879707336
epoch: 18. Loss: 0.5889583230018616
epoch: 19. Loss: 0.5938061475753784
epoch: 20. Loss: 0.4333333373069763
epoch: 21. Loss: 0.4893699586391449
epoch: 22. Loss: 0.5328192114830017
epoch: 23. Loss: 0.823674738407135
epoch: 24. Loss: 0.6617331504821777
epoch: 25. Loss: 0.6422055959701538
epoch: 26. Loss: 0.5801045298576355
epoch: 27. Loss: 0.5938060879707336
epoc

In [22]:
results2_df = pd.DataFrame(results2).T
print(results2_df)

               TP    FP    TN    FN  Precision    Recall  F1-score
Wavelet      40.0  24.0  26.0  43.0      0.625  0.481928  0.544218
FFT          50.0  50.0   0.0  42.0      0.500  0.543478  0.520833
STFT         50.0  50.0   0.0  45.0      0.500  0.526316  0.512821
spectrogram  50.0  50.0   0.0  50.0      0.500  0.500000  0.500000


In [25]:
import importlib
import data_utils  
importlib.reload(data_utils)

ratio=[0.75,0.5,0.25,0.1]
results3 = {}
for r in ratio:
        train_loader, test_loader = create_dataloaders(
                data,
                Lab_Detection,
                batch_size=8,
                test_size=0.1,
                random_state=42,
                shuffle_train=True,
                transform=None,
                Reduce_the_size=r)
        
        model_reduce=MLP(input_dim=train_loader.dataset.data.shape[1], hidden_layers=[2000, 3000], activation="relu", output_dim=output_dim)
        results3[str(r)] = train_model_picking(
                model_reduce,
                train_loader,
                test_loader,
                epochs=100
                )

epoch: 0. Loss: 0.04512203484773636
epoch: 1. Loss: 0.012001107446849346
epoch: 2. Loss: 0.0073437816463410854
epoch: 3. Loss: 0.05087112635374069
epoch: 4. Loss: 0.0019618638325482607
epoch: 5. Loss: 0.0036327815614640713
epoch: 6. Loss: 0.003206561552360654
epoch: 7. Loss: 0.0014126021414995193
epoch: 8. Loss: 0.04482081159949303
epoch: 9. Loss: 0.0014036507345736027
epoch: 10. Loss: 0.004210770130157471
epoch: 11. Loss: 2.703156951611163e-06
epoch: 12. Loss: 0.0013190964236855507
epoch: 13. Loss: 0.0026878626085817814
epoch: 14. Loss: 0.0030138709116727114
epoch: 15. Loss: 0.0014435297343879938
epoch: 16. Loss: 0.002777986926957965
epoch: 17. Loss: 0.0027064387686550617
epoch: 18. Loss: 0.0013170295860618353
epoch: 19. Loss: 0.0025979303754866123
epoch: 20. Loss: 0.0013424595817923546
epoch: 21. Loss: 0.0025989036075770855
epoch: 22. Loss: 0.0013040866469964385
epoch: 23. Loss: 0.0026664813049137592
epoch: 24. Loss: 0.0026242153253406286
epoch: 25. Loss: 0.0012957360595464706
epoch:

In [27]:
results3_df = pd.DataFrame(results3).T
print(results3_df)

        TP    FP    TN    FN  Precision    Recall  F1-score
0.75  32.0  25.0  25.0  43.0   0.561404  0.426667  0.484848
0.5   22.0  20.0  30.0  44.0   0.523810  0.333333  0.407407
0.25  17.0  20.0  30.0  46.0   0.459459  0.269841  0.340000
0.1   21.0  14.0  36.0  46.0   0.600000  0.313433  0.411765
